# Game 8 - The Cross-Modal Truth Arena

**Team:** Ded_Sec

This completed notebook documents the final multimodal system and loads the
generated validation and public-test artifacts. The image expert is the
optimized 160-pixel scratch CNN from Game 6. The text expert is the local
MiniLM encoder plus logistic classifier and quality features from Game 6.
A locally shipped CLIP encoder adds a semantic image-text agreement score
(clip_similarity) so mismatched pairs are detectable even when neither
component appears in the canonical manifests.

The relation layer measures canonical image-text pairing, whether both
components are known, model agreement, CLIP similarity, and the probability
gap between the two modalities. It compares image-only, text-only, weighted
fusion, relation-gated fusion, logistic stacking, gradient-boosted stacking,
and the average of the two stacking models. The stacking models are fit on
the original rows plus two relation-blind copies (one clean, one with CLIP
jitter) so they generalize to hidden tests built from entirely new material.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
comparison = pd.read_csv(ROOT / "game8_model_comparison.csv")
evidence = pd.read_csv(ROOT / "game8_evidence_analysis.csv")
public = pd.read_csv(ROOT / "game8_public_predictions.csv")
comparison.sort_values("macro_f1", ascending=False)

,system,modality,accuracy,macro_f1,precision,recall,relation_strategy,selected
5,relation_gradient_boosting,fusion,0.9340,0.934000,0.934007,0.9340,"nonlinear probability, canonical relation, and...",True
6,relation_stack_average,fusion,0.9308,0.930800,0.930807,0.9308,mean of the logistic and gradient-boosted stac...,False
3,canonical_relation_gate,fusion,0.9248,0.924719,0.926636,0.9248,known canonical mismatch forces fake,False
4,relation_logistic_stack,fusion,0.9224,0.922400,0.922404,0.9224,probabilities + confidence + canonical relatio...,False
2,weighted_probability_fusion,fusion,0.7496,0.744417,0.771632,0.7496,"weighted average, image_weight=0.500",False
0,image_only,image,0.7268,0.723600,0.737813,0.7268,none,False
1,text_only,text,0.6604,0.652412,0.676638,0.6604,none,False


## Selected system

`relation_gradient_boosting` was selected by validation macro-F1.

- Supplied benchmark accuracy: 0.9340
- Supplied benchmark macro-F1: 0.9340
- Image weight in the simple fusion baseline: 0.500

Canonical mismatches are strong evidence for `fake`. For matched pairs, the
fusion model decides how much to trust the image and text experts from their
probabilities, confidence, agreement, and interaction features.

This benchmark score uses all canonical manifests available from earlier games.
It is a competition benchmark result, not a guaranteed accuracy on unrelated
real-world data.


In [2]:
pd.crosstab(
    evidence["primary_evidence"],
    [evidence["correct"], evidence["reviewer_flag"]],
    margins=True,
)

correct             False       True        All
reviewer_flag       False True False True      
primary_evidence                               
image                  26   35   625  287   973
image_text_relation    19    8  1056  102  1185
text                   21   18   144   84   267
uncertain               0   38     0   37    75
All                    66   99  1825  510  2500

In [3]:
errors = evidence.loc[~evidence["correct"]]
errors[[
    "sample_id", "true_label", "predicted_label", "confidence",
    "primary_evidence", "reviewer_flag", "error_category"
]].head(30)

,sample_id,true_label,predicted_label,confidence,primary_evidence,reviewer_flag,error_category
17,g8_validation__000017,fake,real,0.765152,text,False,modality_disagreement_error
20,g8_validation__000020,fake,real,0.954744,text,False,both_experts_wrong
22,g8_validation__000022,real,fake,0.518482,uncertain,True,modality_disagreement_error
27,g8_validation__000027,fake,real,0.885090,image,False,modality_disagreement_error
33,g8_validation__000033,real,fake,0.841893,image,True,modality_disagreement_error
53,g8_validation__000053,real,fake,0.501174,uncertain,True,modality_disagreement_error
54,g8_validation__000054,fake,real,0.958201,image,False,both_experts_wrong
71,g8_validation__000071,fake,real,0.685389,image_text_relation,True,modality_disagreement_error
76,g8_validation__000076,fake,real,0.708359,image_text_relation,False,fusion_overruled_correct_expert
81,g8_validation__000081,fake,real,0.536594,uncertain,True,modality_disagreement_error


In [4]:
required = [
    "sample_id", "predicted_label", "confidence",
    "primary_evidence", "reviewer_flag",
]
assert public.columns.tolist() == required
assert public["primary_evidence"].isin(
    ["image", "text", "image_text_relation", "uncertain"]
).all()
assert public["confidence"].between(0, 1).all()
public.head(10)

,sample_id,predicted_label,confidence,primary_evidence,reviewer_flag
0,g8_public_test__000000,real,0.983364,text,False
1,g8_public_test__000001,real,0.995496,image,False
2,g8_public_test__000002,fake,0.999133,image_text_relation,False
3,g8_public_test__000003,fake,0.997935,image_text_relation,False
4,g8_public_test__000004,real,0.995778,image,False
5,g8_public_test__000005,real,0.995117,image,False
6,g8_public_test__000006,fake,0.998365,image_text_relation,False
7,g8_public_test__000007,fake,0.999014,image_text_relation,True
8,g8_public_test__000008,fake,0.999279,image_text_relation,False
9,g8_public_test__000009,real,0.994299,image,True


## Trust and review policy

Cases are flagged for review when confidence is below 0.68, when the image and
text experts strongly disagree, or when a detected relation mismatch is not
supported with at least 0.90 confidence. The evidence field is intentionally
limited to the four values required by the release contract.
